# $\mathrm{Na}_{\mathrm{V}}1.5$: predicted and experimental distance analysis

Here we compare Cα–Cα distance distributions from AlphaFold2 ensembles with
distances measured in experimental structures. Data are loaded from versioned GitHub
raw URLs so the analysis can run outside the original workstation.

## How to read these plots

- A narrow violin indicates that the predicted ensemble consistently samples a similar geometry.
- An experimental marker inside the main violin density is compatible with the predicted ensemble.
- A marker near a tail indicates partial or rare sampling of the experimental conformation.
- A marker outside the predicted range identifies a systematic structural difference worth inspecting.

Distances alone do **not** constitute a structural superposition. They are invariant to rigid-body
alignment and therefore allow local geometry comparisons without superposing coordinates. A true
superposition analysis (aligned Cα RMSD, aligned residue coverage, and domain-specific RMSD) requires
the predicted and experimental coordinate files, which are not included in this repository. The
distance comparisons below should consequently be interpreted as local structural agreement, not
as a global RMSD analysis.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import sys
repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None
from shared.dataset_selection import distance_csv_options, load_selected_distance_csv


In [ ]:
from shared.presentation_export import install_notebook_figure_export
WRITING_FIGURE_DIR = repo_root.parent / 'vgci_mutants_writing' / 'figures' / 'nav15'
install_notebook_figure_export(WRITING_FIGURE_DIR, 'Nav15_distanceDistribution_vsExperimental')


In [ ]:
# Nav1.5 WT ensembles
# Use the newly generated WT masked tables. Select whether the IFM motif was
# retained at the IFM site. In masked_v2_noIFM, the native IFM sequence remains
# present and the IFM residues are excluded from the masked region, so their MSA
# information is retained. masked_v2 is the default broader-mask comparison.
DATASET_SELECTION = 'all_ok_3'
data = repo_root / 'nav15' / 'dataDistances'
DATASET_PATHS = {
    'WT vanilla': distance_csv_options(repo_root, data / '26-07-27_Nav15_wt_vanillaAF2_distances_all_all.csv', 'Nav15', 'WT', 'vanilla'),
    'WT masked_v2': distance_csv_options(repo_root, data / '26-07-27_Nav15_wt_maskedv2_AF2_distances_all_all.csv', 'Nav15', 'WT', 'masked'),
    'WT masked_v2_noIFM': distance_csv_options(repo_root, data / '26-07-27_Nav15_wt_maskedv2_noIFM_AF2_distances_all_all.csv', 'Nav15', 'WT', 'masked'),
    'QQQ vanilla': distance_csv_options(repo_root, data / '26-07-27_Nav15_qqq_vanilla_AF2_distances_all_all.csv', 'Nav15', 'QQQ', 'vanilla'),
    'QQQ masked': distance_csv_options(repo_root, data / '26-07-27_Nav15_qqq_masked_AF2_distances_all_all.csv', 'Nav15', 'QQQ', 'masked'),
    'QQQ masked_v2': distance_csv_options(repo_root, data / '26-07-27_Nav15_qqq_maskedv2_AF2_distances_all_all.csv', 'Nav15', 'QQQ', 'masked'),
}
DATASET_PATHS
df_vanilla = load_selected_distance_csv('WT vanilla', DATASET_PATHS['WT vanilla'], DATASET_SELECTION)
df_masks = {
    variant: load_selected_distance_csv(f'WT {variant}', DATASET_PATHS[f'WT {variant}'], DATASET_SELECTION)
    for variant in ('masked_v2', 'masked_v2_noIFM')
}
df_mask = df_masks['masked_v2']  # Preserve the existing masked plots.
df_qqq = {
    protocol: load_selected_distance_csv(f'QQQ {protocol}', DATASET_PATHS[f'QQQ {protocol}'], DATASET_SELECTION)
    for protocol in ('vanilla', 'masked', 'masked_v2')
}
print(f"Loaded WT: {len(df_vanilla)} vanilla, {len(df_masks['masked_v2'])} masked v2, {len(df_masks['masked_v2_noIFM'])} masked v2 no-IFM")
print(f"Loaded IFM→QQQ: {len(df_qqq['vanilla'])} vanilla, {len(df_qqq['masked'])} masked, {len(df_qqq['masked_v2'])} masked v2")


In [ ]:
df_mask

## Experimental-distance comparisons

The selectivity filter, intracellular gate, and voltage sensors are evaluated against PDB-specific
experimental references. Vanilla and masked ensembles are considered together when distinguishing
protocol-dependent changes from the intrinsic spread of each ensemble.


In [ ]:
from shared.plotting import plot_distances_by_alias_violin


In [ ]:
# Additional Nav1.5 experimental state distances, mapped by seven-residue motifs.
# The model/AF2 labels are retained as dictionary keys; PDB residue mapping is explicit below.
from shared.plotting import NAV15_PALETTE, NAV15_EXPERIMENTAL_STYLES, NAV15_EXPERIMENTAL_STYLES, NAV15_EXPERIMENTAL_STYLES, NAV15_EXPERIMENTAL_STYLES, NAV15_EXPERIMENTAL_STYLES, NAV15_EXPERIMENTAL_STYLES
NAV15_8VY_EXPERIMENTAL = {
    "8VYJ": {
        "state": "Nav1.5 Class I",
        "pdb_residue_map": {
            "D373": 372, "E704": 898, "K1103": 1419, "A1395": 1711,
            "M415": 414, "A742": 936, "I1154": 1470, "I1455": 1771,
            "F165": 164, "R220": 219, "R229": 228, "F560": 756,
            "R612": 808, "K624": 820, "F934": 1250, "R984": 1303,
            "K1000": 1316, "F1251": 1567, "R1307": 1623, "R1322": 1638,
        },
        "selectivity_filter": {
            "D373-E704": [6.252], "D373-K1103": [11.170], "D373-A1395": [8.817],
            "E704-K1103": [10.645], "E704-A1395": [12.573], "K1103-A1395": [7.742],
        },
        "intracellular_gate": {
            "M415-A742": [12.081], "M415-I1154": [15.763], "M415-I1455": [10.592],
            "A742-I1154": [15.214], "A742-I1455": [17.324], "I1154-I1455": [11.379],
        },
        "vsd": {
            "F165-R220": [15.765], "F165-R229": [14.003], "F560-R612": [14.339],
            "F560-K624": [19.779], "F934-R984": [17.544], "F934-K1000": [15.081],
            "F1251-R1307": [19.645], "F1251-R1322": [20.126],
        },
    },
    "8VYK": {
        "state": "Nav1.5 Class II",
        "pdb_residue_map": {
            "D373": 372, "E704": 898, "K1103": 1419, "A1395": 1711,
            "M415": 414, "A742": 936, "I1154": 1470, "I1455": 1771,
            "F165": 164, "R220": 219, "R229": 228, "F560": 756,
            "R612": 808, "K624": 820, "F934": 1250, "R984": 1303,
            "K1000": 1316, "F1251": 1567, "R1307": 1623, "R1322": 1638,
        },
        "selectivity_filter": {
            "D373-E704": [6.131], "D373-K1103": [10.598], "D373-A1395": [8.243],
            "E704-K1103": [10.589], "E704-A1395": [12.419], "K1103-A1395": [7.787],
        },
        "intracellular_gate": {
            "M415-A742": [11.921], "M415-I1154": [16.525], "M415-I1455": [11.259],
            "A742-I1154": [15.803], "A742-I1455": [17.562], "I1154-I1455": [11.877],
        },
        "vsd": {
            "F165-R220": [15.878], "F165-R229": [14.183], "F560-R612": [13.910],
            "F560-K624": [20.158], "F934-R984": [17.304], "F934-K1000": [15.150],
            "F1251-R1307": [19.708], "F1251-R1322": [19.893],
        },
    },
}

NAV15_EXPERIMENTAL_COLORS = [NAV15_EXPERIMENTAL_STYLES[pdb]['color'] for pdb in ('7FBS','6UZ3','8VYJ','8VYK')]

from functools import partial
from shared.experimental_overlays import (
    nav15_state_experimentals as _nav15_state_experimentals,
)
add_nav15_state_experimentals = partial(
    _nav15_state_experimentals,
    experimental_states=NAV15_8VY_EXPERIMENTAL,
    colors=NAV15_EXPERIMENTAL_COLORS[2:],
)


In [ ]:
# Selectvity Filter Check
'''   ("ASP", 373, "GLU", 704 ), #Selectivity filter
    ("ASP", 373, "LYS", 1103 ),
    ("ASP", 373, "ALA", 1395 ),
    ("GLU", 704, "LYS", 1103 ),
    ("GLU", 704, "ALA", 1395 ),
    ("LYS", 1103, "ALA", 1395 )'''


alias_dict = {
    'D373-E704': 'CA_ASP373_CA-GLU704_CA',
    'D373-K1103': 'CA_ASP373_CA-LYS1103_CA',
    'D373-A1395': 'CA_ASP373_CA-ALA1395_CA',
    'E704-K1103': 'CA_GLU704_CA-LYS1103_CA',
    'E704-A1395': 'CA_GLU704_CA-ALA1395_CA',
    'K1103-A1395': 'CA_LYS1103_CA-ALA1395_CA'
}
SELECTIVITY_FILTER_ALIASES = alias_dict.copy()

# Colors distinguish the six selectivity-filter pairs.
custom_colors = ['#53acc9', '#89d7eb', '#91c7f9', '#53acc9', '#4dabae', '#9370DB']

# Experimental distances to overlay (multiple datasets with lists for each alias)
exp_distances_list = [
    { #7fbs #D373,901,1421,1713
        'D373-E704': [5.804],
        'D373-K1103': [10.912],
        'D373-A1395': [8.988],
        'E704-K1103': [10.342],
        'E704-A1395': [12.075],
        'K1103-A1395': [6.84]
    },
    { #6uz3 #D373,E901,1421,1713
        'D373-E704': [5.332],
        'D373-K1103': [9.773],
        'D373-A1395': [8.78],
        'E704-K1103': [9.433],
        'E704-A1395': [11.71],
        'K1103-A1395': [6.371]
    }
]

# Labels for the datasets
dataset_labels = ['Experimental | PDB ID: 7FBS — QQQ mutant', 'Experimental | PDB ID: 6UZ3 — WT'] 
# Add motif-mapped 8VYJ/8VYK state measurements to the older 7FBS/6UZ3 dictionary.
_new_distances, _new_labels, _new_colors = add_nav15_state_experimentals("selectivity_filter")
exp_distances_list.extend(_new_distances)
dataset_labels.extend(_new_labels)
# Custom colors for each PDB dataset
pdb_colors = NAV15_EXPERIMENTAL_COLORS[:2]

pdb_colors = list(pdb_colors[:2]) + _new_colors

# WT masked-v2 selectivity-filter distribution with experimental references.
#plot_distances_by_alias_violin(df, alias_dict, xp_distances_list=exp_distances_list, title_custom_add='Selectivity Filter Subsampling 32-64', colors=[NAV15_PALETTE['WT_MASKED_V2']] * len(alias_dict), fig_width=14, fig_height=8, dataset_labels=dataset_labels, pdb_colors=pdb_colors)
plot_distances_by_alias_violin(df_mask, alias_dict, exp_distances_list=exp_distances_list, title_custom_add='Nav1.5 | WT | masked v2 | Selectivity filter', colors=[NAV15_PALETTE['WT_MASKED_V2']] * len(alias_dict), fig_width=6, fig_height=6, dataset_labels=dataset_labels, pdb_colors=pdb_colors)


In [ ]:
plot_distances_by_alias_violin(df_vanilla, alias_dict, exp_distances_list=exp_distances_list, title_custom_add='Nav1.5 | WT | vanilla | Selectivity filter', colors=[NAV15_PALETTE['WT_VAN']] * len(alias_dict), fig_width=6, fig_height=6, dataset_labels=dataset_labels, pdb_colors=pdb_colors)


In [ ]:
# Intracellular-gate distance set.
alias_dict = {
    'M415-A742':'CA_MET415_CA-ALA742_CA',
    'M415-I1154':'CA_MET415_CA-ILE1154_CA',
    'M415-I1455':'CA_MET415_CA-ILE1455_CA',
    'A742-I1154':'CA_ALA742_CA-ILE1154_CA',
    'A742-I1455':'CA_ALA742_CA-ILE1455_CA',
    'I1154-I1455':'CA_ILE1154_CA-ILE1455_CA'
}
INTRACELLULAR_GATE_ALIASES = alias_dict.copy()

# A related color family keeps the six gate pairs visually grouped.
custom_colors = ['#eb5c9e', '#ea96a3', '#e689e6', '#e891c3', '#d057a8', '#e1818f']

# Experimental distances to overlay (example values, with lists for each alias)
exp_distances_list = [
    { #7FBS #M415,A939,I1472,I1773
    'M415-A742':[11.734],
    'M415-I1154':[15.009],
    'M415-I1455':[12.222],
    'A742-I1154':[14.315],
    'A742-I1455':[18.59],
    'I1154-I1455':[10.329]
}, { #6UZ3
    'M415-A742':[10.933],
    'M415-I1154':[14.410],
    'M415-I1455':[10.041],
    'A742-I1154':[14.044],
    'A742-I1455':[16.527],
    'I1154-I1455':[10.548]
}
]
# Labels for the datasets
dataset_labels = ['Experimental | PDB ID: 7FBS — QQQ mutant', 'Experimental | PDB ID: 6UZ3 — WT']

# Add motif-mapped 8VYJ/8VYK state measurements to the older 7FBS/6UZ3 dictionary.
_new_distances, _new_labels, _new_colors = add_nav15_state_experimentals("intracellular_gate")
exp_distances_list.extend(_new_distances)
dataset_labels.extend(_new_labels)
# Custom colors for each PDB dataset
pdb_colors = NAV15_EXPERIMENTAL_COLORS[:2]

pdb_colors = list(pdb_colors[:2]) + _new_colors

# WT masked-v2 gate distribution with experimental references.
plot_distances_by_alias_violin(df_mask, alias_dict, exp_distances_list=exp_distances_list, title_custom_add='Nav1.5 | WT | masked v2 | Intracellular gate', colors=[NAV15_PALETTE['WT_MASKED_V2']] * len(alias_dict), fig_width=6, fig_height=6, dataset_labels=dataset_labels, pdb_colors=pdb_colors)


In [ ]:
plot_distances_by_alias_violin(df_vanilla, alias_dict, exp_distances_list=exp_distances_list, title_custom_add='Nav1.5 | WT | vanilla | Intracellular gate', colors=[NAV15_PALETTE['WT_VAN']] * len(alias_dict), fig_width=6, fig_height=6, dataset_labels=dataset_labels, pdb_colors=pdb_colors)


In [ ]:
# Gating charges vs the aromatic middle in S2
alias_dict = {
    'F165-R220': 'CA_ARG220_CA-PHE165_CA',
    'F165-R229': 'CA_LYS229_CA-PHE165_CA',
    'F560-R612': 'CA_ARG612_CA-PHE560_CA',
    'F560-K624': 'CA_LYS624_CA-PHE560_CA',
    'F934-R984': 'CA_LYS984_CA-PHE934_CA',
    'F934-K1000': 'CA_ARG1000_CA-PHE934_CA',
    'F1251-R1307': 'CA_ARG1307_CA-PHE1251_CA',
    'F1251-R1322': 'CA_ARG1322_CA-PHE1251_CA',
}
VOLTAGE_SENSOR_ALIASES = alias_dict.copy()

# Colors distinguish the voltage-sensor measurements.
custom_colors = ['#9fa046','#92a348', '#7aaa48','#6aae52', '#4fb38c','#4aac99','#43b5b1','#3ebbbd']

# Experimental distances to overlay (multiple datasets with lists for each alias)
exp_distances_list = [
    { #7fbs
    'F165-R220': [14.115], #F165
    'F165-R229': [12.996], #
    'F560-R612': [13.47],
    'F560-K624': [20.356],
    'F934-R984': [16.317], #r1305 F1252
    'F934-K1000': [14.43], #1318
    'F1251-R1307': [19.02],
    'F1251-R1322': [18.533]
   },
    { #6uz3
    'F165-R220': [14.614], #
    'F165-R229': [13.695], #
    'F560-R612': [13.199], #R809 F757
    'F560-K624': [21.96], #K821
    'F934-R984': [18.27], 
    'F934-K1000': [14.585],
    'F1251-R1307': [19.207],#F1569 R1625
    'F1251-R1322': [19.19] #F1569 R1640
    }
]

# Labels for the datasets
dataset_labels = ['Experimental | PDB ID: 7FBS — QQQ mutant', 'Experimental | PDB ID: 6UZ3 — WT']

# Add motif-mapped 8VYJ/8VYK state measurements to the older 7FBS/6UZ3 dictionary.
_new_distances, _new_labels, _new_colors = add_nav15_state_experimentals("vsd")
exp_distances_list.extend(_new_distances)
dataset_labels.extend(_new_labels)
# Custom colors for each PDB dataset
pdb_colors = NAV15_EXPERIMENTAL_COLORS[:2]

pdb_colors = list(pdb_colors[:2]) + _new_colors

# WT masked-v2 voltage-sensor distribution with experimental references.
plot_distances_by_alias_violin(df_mask, alias_dict, exp_distances_list=exp_distances_list, title_custom_add='Nav1.5 | WT | masked v2 | Voltage sensor | experimental distances', colors=[NAV15_PALETTE['WT_MASKED_V2']] * len(alias_dict), fig_width=6, fig_height=6, dataset_labels=dataset_labels, pdb_colors=pdb_colors)


In [ ]:
from shared.plotting import plot_split_violin


## WT–mutant and masked–vanilla split comparisons

Transparent superpositions are omitted because overlapping violin bodies obscure the two distributions.


## Near-mutation distance distributions


In [ ]:
## Nav1.5 mutation-region correspondence distances
# Use the model residue correspondences already represented by the plotted
# functional landmarks; no broad numeric alias search is used.
NAV15_MUTATION_CORRESPONDENCE = {
    "selectivity_filter": ["D373-E704", "D373-K1103", "D373-A1395", "E704-K1103", "E704-A1395", "K1103-A1395"],
    "intracellular_gate": ["M415-A742", "M415-I1154", "M415-I1455", "A742-I1154", "A742-I1455", "I1154-I1455"],
    "voltage_sensor": ["F165-R220", "F165-R229", "F560-R612", "F560-K624", "F934-R984", "F934-K1000", "F1251-R1307", "F1251-R1322"],
}
nav_mutation_aliases = {a: alias_dict[a] for group in NAV15_MUTATION_CORRESPONDENCE.values() for a in group if a in alias_dict}
print("Nav1.5 correspondence aliases available in this region:", list(nav_mutation_aliases))

# These aliases are retained for auditing; ensemble comparisons use the experimental split violins below.


## $\mathrm{Na}_{\mathrm{V}}1.5$ experimental correspondence validation


In [ ]:
from shared.structure_distances import read_ca_atoms as nav_ca, ca_distance_map as nav_distances

## Double recalculation audit of all plotted Nav1.5 experimentals
import math, re


# These are the explicit PDB residue mappings used for the plotted aliases.
NAV15_PDB_MAP = {
 "D373-E704":("A",373,"A",901), "D373-K1103":("A",373,"A",1421), "D373-A1395":("A",373,"A",1713),
 "M415-A742":("A",415,"A",939), "M415-I1154":("A",415,"A",1472), "M415-I1455":("A",415,"A",1773),
 "F165-R220":("A",165,"A",220), "F165-R229":("A",165,"A",229), "F560-R612":("A",757,"A",809), "F560-K624":("A",757,"A",821),
}
for pdb in ("7FBS","6UZ3"):
    first,missing1=nav_distances(f"experimental/{pdb}.pdb",NAV15_PDB_MAP)
    second,missing2=nav_distances(f"experimental/{pdb}.pdb",NAV15_PDB_MAP)
    print(pdb,"pass1",first,"pass2 identical:",first==second,"missing:",missing1 or missing2)


## Experimental marker encoding


In [ ]:
## Nav1.5 experimental-marker visibility check
# Replot the current voltage-sensor aliases with explicit experimental labels.
plot_distances_by_alias_violin(
    df_mask, alias_dict, exp_distances_list=exp_distances_list,
    title_custom_add="Nav1.5 | WT | masked v2 | Voltage sensor + experimental",
    colors=[NAV15_PALETTE['WT_MASKED_V2']] * len(alias_dict), fig_width=10.5, fig_height=8,
    dataset_labels=dataset_labels,
    pdb_colors=NAV15_EXPERIMENTAL_COLORS,
)


## Additional split-violin views


In [ ]:
from shared.plotting import plot_split_comparison_with_experimentals
from shared.experimental_overlays import experimental_rows

## Split violins with experimental structures
# The legacy 1,200-row WT-masked table is excluded because it has no exact
# overlap with an RMSD manifest.
COMPARISON_REGIONS = {
    'selectivity_filter': ('Selectivity filter', SELECTIVITY_FILTER_ALIASES),
    'intracellular_gate': ('Intracellular gate', INTRACELLULAR_GATE_ALIASES),
    'vsds': ('Voltage sensor', VOLTAGE_SENSOR_ALIASES),
}
PAIRWISE_COMPARISONS = [
    ('WT | vanilla', df_vanilla, 'WT | masked v2', df_masks['masked_v2'], NAV15_PALETTE['WT_VAN'], NAV15_PALETTE['WT_MASKED_V2']),
    ('WT | vanilla', df_vanilla, 'WT | masked v2 no-IFM', df_masks['masked_v2_noIFM'], NAV15_PALETTE['WT_VAN'], NAV15_PALETTE['WT_MASKED_V2_NOIFM']),
    ('WT | masked v2', df_masks['masked_v2'], 'WT | masked v2 no-IFM', df_masks['masked_v2_noIFM'], NAV15_PALETTE['WT_MASKED_V2'], NAV15_PALETTE['WT_MASKED_V2_NOIFM']),
    ('IFM→QQQ | vanilla', df_qqq['vanilla'], 'IFM→QQQ | masked', df_qqq['masked'], NAV15_PALETTE['QQQ_VAN'], NAV15_PALETTE['QQQ_MASKED']),
    ('IFM→QQQ | vanilla', df_qqq['vanilla'], 'IFM→QQQ | masked v2', df_qqq['masked_v2'], NAV15_PALETTE['QQQ_VAN'], NAV15_PALETTE['QQQ_MASKED_V2']),
    ('IFM→QQQ | masked', df_qqq['masked'], 'IFM→QQQ | masked v2', df_qqq['masked_v2'], NAV15_PALETTE['QQQ_MASKED'], NAV15_PALETTE['QQQ_MASKED_V2']),
    ('WT | vanilla', df_vanilla, 'IFM→QQQ | vanilla', df_qqq['vanilla'], NAV15_PALETTE['WT_VAN'], NAV15_PALETTE['QQQ_VAN']),
    ('WT | masked v2', df_masks['masked_v2'], 'IFM→QQQ | masked v2', df_qqq['masked_v2'], NAV15_PALETTE['WT_MASKED_V2'], NAV15_PALETTE['QQQ_MASKED_V2']),
]
for region_key, (region_label, aliases) in COMPARISON_REGIONS.items():
    references = experimental_rows(repo_root, 'Nav1.5', region_key, aliases)
    for label1, frame1, label2, frame2, color1, color2 in PAIRWISE_COMPARISONS:
        plot_split_comparison_with_experimentals(
            frame1, frame2, aliases, label1, label2,
            f'Nav1.5 | {label1} vs {label2} | {region_label} | experimental distances',
            [color1, color2], references,
        )

## Gate openness and coordinated IFM-engagement sampling
# The mean of the two sequence-verified engagement contacts is used as a compact
# engagement coordinate: larger values indicate a more disengaged IFM/QQQ motif.
from shared.channel_state_analysis import plot_gate_state_comparison
wt_latch_columns = ['CA_PHE1170_CA-ASN1343_CA', 'CA_PHE1170_CA-ASN1449_CA']
qqq_latch_columns = ['CA_GLN1170_CA-ASN1343_CA', 'CA_GLN1170_CA-ASN1449_CA']
nav_gate_state_summary = []
for protocol, wt_frame, qqq_frame, colors in [
    ('vanilla', df_vanilla, df_qqq['vanilla'], [NAV15_PALETTE['WT_VAN'], NAV15_PALETTE['QQQ_VAN']]),
    ('masked v2', df_masks['masked_v2'], df_qqq['masked_v2'], [NAV15_PALETTE['WT_MASKED_V2'], NAV15_PALETTE['QQQ_MASKED_V2']]),
]:
    summary, _ = plot_gate_state_comparison(
        wt_frame, qqq_frame, INTRACELLULAR_GATE_ALIASES,
        wt_latch_columns, qqq_latch_columns,
        'Nav1.5', 'IFM→QQQ', protocol,
        'Mean IFM/QQQ–N1659/N1765 engagement distance (Å)', colors,
    )
    nav_gate_state_summary.append(summary)
nav_gate_state_summary = pd.concat(nav_gate_state_summary, ignore_index=True)
display(nav_gate_state_summary.round(3))

## WT-versus-mutant voltage-sensor bias in both protocols
from shared.channel_state_analysis import plot_vsd_bias
nav_vsd_bias, _ = plot_vsd_bias(
    {'vanilla': (df_vanilla, df_qqq['vanilla']), 'masked v2': (df_masks['masked_v2'], df_qqq['masked_v2'])},
    VOLTAGE_SENSOR_ALIASES, 'Nav1.5', 'IFM→QQQ',
    {'vanilla': NAV15_PALETTE['QQQ_VAN'], 'masked v2': NAV15_PALETTE['QQQ_MASKED_V2']},
)
display(nav_vsd_bias.round(3))
